# WB2 baselines on a real target (Kaggle / Colab)

**What this notebook produces** (everything lands in `OUT_DIR` and is zipped at the end):

| Step | Output | Rough time on a T4 |
|---|---|---|
| 3–4 | Ollama + target model, classifier service, both health-checked | 10 min |
| 5 | One smoke round through every defense | 2 min |
| 6 | **Pilot**: 32 campaigns, measured seconds per round | 20–40 min |
| 7 | **Classifier-only baseline**: TPR/FPR of the classifier on our payloads and hard negatives | 5 min |
| 8 | Budget/seeds chosen by `rtbt.sizing.fit_to_hours` from the pilot's measured throughput | instant |
| 9 | **B1 static baseline**: every scenario × 4 defenses × 2 static attackers vs. the real target | sized to `GPU_HOURS` |
| 10 | Summary tables (`wb2_results_summary.md`), CSV, JSON, the SQLite run store, all zipped | 1 min |

**Before you start.** Kaggle: *Settings → Accelerator → GPU T4 x2* and *Internet → On*. Colab: *Runtime → Change runtime type → T4 GPU*.
If the session dies, just run all cells again: step 9 resumes from the run store and never repeats a finished campaign.

Every number this notebook prints is from a real target model, so campaigns are labelled `measured_result` by the harness itself. Nothing here is typed in by hand.

In [ ]:
# ---- 1. Settings: the only cell you should need to edit ----------------------------------------
REPO_URL   = "https://github.com/yashaswinidinesh/cmpe295a-red-team-blue-team-prototype.git"
BRANCH     = "main"                  # use "wb2-progress" if the WB2 branch is not merged yet
TARGET     = "qwen2.5:7b-instruct"   # needs native tool calling (Ollama 'tools' capability)
CLASSIFIER = "protectai/deberta-v3-base-prompt-injection-v2"   # ungated fallback
PROMPT_GUARD = "meta-llama/Llama-Prompt-Guard-2-86M"           # primary; gated: needs HF_TOKEN + licence accepted
HF_TOKEN   = ""                      # optional; with it the primary classifier is used
GPU_HOURS  = 8.0                     # GPU hours you are willing to spend on step 9 in THIS session
import os, pathlib
OUT_DIR = pathlib.Path("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content") / "rtbt_wb2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR)

In [ ]:
# ---- 2. Code ------------------------------------------------------------------------------------
import subprocess, sys
REPO = pathlib.Path("/tmp/rtbt")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path[:0] = [str(REPO / "src"), str(REPO / "services")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44"], check=True)  # torch is preinstalled
print("repo at", COMMIT)

In [ ]:
# ---- 3. Ollama + target model -------------------------------------------------------------------
import time, json, urllib.request, shutil
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="24h")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(OUT_DIR / "ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
subprocess.run(["ollama", "pull", TARGET], env=env, check=True)
tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))
TARGET_DIGEST = next(m["digest"] for m in tags["models"] if m["name"] == TARGET or m["model"] == TARGET)
print(TARGET, "digest", TARGET_DIGEST[:16])

In [ ]:
# ---- 4. Classifier service (separate process, localhost only) -----------------------------------
model = CLASSIFIER
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    model = PROMPT_GUARD
clf = subprocess.Popen([sys.executable, str(REPO / "services" / "classifier_server.py"), "--model", model, "--port", "8001"],
                       env=dict(os.environ), stdout=open(OUT_DIR / "classifier.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        HEALTH = json.load(urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2)); break
    except Exception:
        time.sleep(1)
        if clf.poll() is not None:
            raise RuntimeError("classifier service exited; see classifier.log (gated model without licence/token?)")
print(HEALTH)

In [ ]:
# ---- 5. Smoke test: one round per defense -------------------------------------------------------
from rtbt.config_io import config_from_dict
from rtbt.harness import ExperimentRunner
from rtbt.scenario_registry import load_suite
SUITE = load_suite(REPO / "scenarios")

def cfg_from(name, **override):
    raw = json.loads((REPO / "configs" / name).read_text())
    raw["real_target_config"]["name"] = TARGET
    raw["real_target_config"]["revision"] = TARGET_DIGEST
    raw["classifier_spec"]["model_name"] = HEALTH["model"]
    raw.update(override)
    return raw

smoke = config_from_dict(cfg_from("kaggle_pilot.json", scenario_ids=["email-summary-exfil"], attack_modes=["static"],
                                  attack_budget=1, seeds=[1]))
rep = ExperimentRunner(suite=SUITE, config=smoke).run()
for c in rep.campaigns:
    bad = [r.error_detail for r in c.rounds if not r.valid]
    print(f"{c.defense_id:16} valid={sum(r.valid for r in c.rounds)}/{len(c.rounds)} evidence={c.evidence_status}", bad[:1])
assert all(r.valid for c in rep.campaigns for r in c.rounds), "fix the errors above before spending GPU time"

In [ ]:
# ---- 6. Pilot: measure seconds per round --------------------------------------------------------
from datetime import datetime
DB = OUT_DIR / "runs.db"
pilot_cfg = OUT_DIR / "pilot_config.json"
pilot_cfg.write_text(json.dumps(cfg_from("kaggle_pilot.json"), indent=2))
from rtbt.persistence import RunStore, rebuild_report

def run_or_resume(run_id, config_path):
    # new run, or finish a stored unfinished one, or nothing if it is already complete
    stored = RunStore(DB).get_run(run_id)
    if stored is None:
        cmd = ["--config", str(config_path), "--run-id", run_id]
    elif stored["status"] != "complete":
        cmd = ["--resume", run_id]
    else:
        print(run_id, "already complete"); return
    p = subprocess.run([sys.executable, str(REPO / "run_demo.py"), *cmd, "--db", str(DB), "--out-dir", str(OUT_DIR)],
                       capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-2000:])
    if p.returncode:
        raise RuntimeError(f"{run_id} stopped (exit {p.returncode}); re-run this cell to resume")

run_or_resume("pilot", pilot_cfg)
pilot = rebuild_report(RunStore(DB), "pilot")
def secs(x): return datetime.fromisoformat(x.replace("Z", "+00:00")).timestamp()
durations = [secs(r.ended_at) - secs(r.started_at) for c in pilot.campaigns for r in c.rounds if r.valid]
SECONDS_PER_ROUND = sorted(durations)[int(0.9 * (len(durations) - 1))]   # p90, not mean: plan for slow rounds
print(f"{len(durations)} rounds; mean {sum(durations)/len(durations):.1f}s, p90 {SECONDS_PER_ROUND:.1f}s per round")

In [ ]:
# ---- 7. Classifier-only baseline (no LLM in the loop) -------------------------------------------
# Positives: every static and published payload rendered for every attackable scenario, inserted
# exactly where the harness inserts it (the read tool's output for the injected item).
# Negatives: every benign hard-negative variant, inserted the same way.
from rtbt.attackers import make_attacker
from rtbt.environments import environment_class
from rtbt.defenses import HttpClassifierAdapter
adapter = HttpClassifierAdapter("http://127.0.0.1:8001/score", HEALTH["model"], allowlist=("127.0.0.1",))

def rendered_item(sc, text):
    env = environment_class(sc.environment)(sc.initial_state, sc.granted_scopes, seed=1)
    env.inject(sc.injection["slot"], text, sc.injection["placement"])
    step = sc.simulated_behavior["read_plan"][-1]
    return env.call(step["tool"], step.get("args", {}))

rows = []
for sc in SUITE.scenarios:
    if sc.is_attackable:
        for mode in ("static", "static_published"):
            a = make_attacker(mode, seed=1)
            for _ in range(10):
                att = a.next_attempt(sc.attack_brief())
                rows.append((sc.id, mode, att.strategy_id, 1, adapter.score(rendered_item(sc, att.payload))[0]))
    for v in sc.benign_variants:
        rows.append((sc.id, "benign", v["id"], 0, adapter.score(rendered_item(sc, v["text"]))[0]))

def rates(th):
    tp = sum(1 for r in rows if r[3] == 1 and r[4] >= th); pos = sum(r[3] for r in rows)
    fp = sum(1 for r in rows if r[3] == 0 and r[4] >= th); neg = len(rows) - pos
    return tp / pos, fp / neg, pos, neg
CLF_TABLE = [(th, *rates(th)) for th in (0.3, 0.5, 0.7, 0.9)]
for th, tpr, fpr, pos, neg in CLF_TABLE:
    print(f"threshold {th}: TPR {tpr:.3f} (n={pos})  FPR {fpr:.3f} (n={neg})")
by_family = {}
for _, mode, strat, y, p in rows:
    if y: by_family.setdefault(strat, []).append(p >= 0.5)
FAMILY_TABLE = sorted((k, sum(v) / len(v), len(v)) for k, v in by_family.items())
import csv
with open(OUT_DIR / "classifier_only.csv", "w", newline="") as f:
    csv.writer(f).writerows([("scenario", "source", "item", "is_attack", "p_injection"), *rows])

In [ ]:
# ---- 8. Size B1 to the measured throughput -------------------------------------------------------
from rtbt.sizing import fit_to_hours, sessions_needed
options = fit_to_hours(SUITE, scenario_ids=None, defenses=4, attack_modes=2,
                       gpu_hours_available=GPU_HOURS, seconds_per_round=SECONDS_PER_ROUND)
for o in options[:5]:
    print(o)
if not options:
    raise SystemExit(f"Even 3 seeds x budget 10 does not fit {GPU_HOURS} GPU h at {SECONDS_PER_ROUND:.1f}s/round. "
                     "Raise GPU_HOURS and run B1 across sessions (it resumes), or cut scenarios.")
CHOSEN = options[0]
print("chosen:", CHOSEN["seeds"], "seeds x budget", CHOSEN["budget"], "->", CHOSEN["gpu_hours"], "GPU h")

In [ ]:
# ---- 9. B1 static baseline (resumable: re-run this cell after a disconnect) ----------------------
b1_cfg = OUT_DIR / "b1_config.json"
b1 = cfg_from("kaggle_b1_static_baseline.json", attack_budget=CHOSEN["budget"], seeds=list(range(1, CHOSEN["seeds"] + 1)))
if RunStore(DB).get_run("b1") is None:
    b1_cfg.write_text(json.dumps(b1, indent=2))   # never change the config of a run already started
run_or_resume("b1", b1_cfg)
print(RunStore(DB).get_run("b1")["status"])

In [ ]:
# ---- 10. Summary + export -----------------------------------------------------------------------
from rtbt.statistics import summarize
b1_report = rebuild_report(RunStore(DB), "b1")
lines = [f"# WB2 real-target results", "",
         f"- commit `{COMMIT}`; target `{TARGET}` digest `{TARGET_DIGEST[:16]}`; classifier `{HEALTH['model']}` rev `{HEALTH.get('revision')}`",
         f"- pilot: {len(durations)} rounds, p90 {SECONDS_PER_ROUND:.1f}s/round; B1: {CHOSEN['seeds']} seeds x budget {CHOSEN['budget']}, "
         f"{len(b1_report.campaigns)} campaigns, evidence: " +
         ", ".join(f"{k}={v}" for k, v in sorted(__import__('collections').Counter(c.evidence_status for c in b1_report.campaigns).items())),
         "", "## B1: mean over scenarios and seeds (95% bootstrap CI)", "",
         "| Attack | Defense | ASR | Legit task completion | Benign block | n campaigns |", "|---|---|---|---|---|---|"]
for mode in ("static", "static_published"):
    for d in ("baseline", "defensive_prompt", "classifier", "tool_validator"):
        cs = [c for c in b1_report.primary_campaigns if c.attack_mode == mode and c.defense_id == d]
        pooled = summarize(cs)["pooled"]
        s = pooled[0]["stats"] if pooled else {}
        def f(m):
            a = s.get(m) or {}
            return "n/a" if a.get("mean") is None else f"{a['mean']:.3f} [{a['ci_95_low']:.3f}, {a['ci_95_high']:.3f}]"
        lines.append(f"| {mode} | {d} | {f('campaign_average_asr')} | {f('legitimate_task_completion_rate')} | {f('benign_block_rate')} | {len(cs)} |")
lines += ["", "## Classifier only (no LLM)", "", "| Threshold | TPR | FPR | n attack | n benign |", "|---|---|---|---|---|"]
lines += [f"| {th} | {tpr:.3f} | {fpr:.3f} | {pos} | {neg} |" for th, tpr, fpr, pos, neg in CLF_TABLE]
lines += ["", "| Payload family | Detected at 0.5 | n |", "|---|---|---|"] + [f"| {k} | {v:.3f} | {n} |" for k, v, n in FAMILY_TABLE]
(OUT_DIR / "wb2_results_summary.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
shutil.make_archive(str(OUT_DIR.parent / "rtbt_wb2_results"), "zip", OUT_DIR)
print("\nDownload:", OUT_DIR.parent / "rtbt_wb2_results.zip")